# 📈 Rendements, volatilité, corrélation et Markowitz

| # | Section | Question |
|---|---|---|
| 1 | Prix | Comment comparer des actions ? |
| 2 | Rendements | De combien le prix a-t-il bougé ? |
| 3 | Log-rendements | Comment additionner des rendements dans le temps ? |
| 4 | Volatilité | À quel point ça bouge ? |
| 5 | Corrélation | Est-ce que les actions bougent ensemble ? |
| 6 | Markowitz | Comment combiner les actions ? |
| 7 | Sharpe et Sortino | Quel portefeuille est le meilleur ? |

▶️ **Exécution → Tout exécuter**

## 0. Préparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import yfinance as yf

# Style de tous les graphiques
COULEURS = ["#2E86AB", "#E4572E", "#3BB273", "#F2A541"]
plt.rcParams.update({
    "figure.figsize": (14, 7),
    "font.size": 13,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=COULEURS),
})

def pourcent(axes="y", ax=None):
    """Affiche l'axe x et/ou y en %."""
    ax = ax or plt.gca()
    if "x" in axes: ax.xaxis.set_major_formatter(PercentFormatter(1))
    if "y" in axes: ax.yaxis.set_major_formatter(PercentFormatter(1))

JOURS_PAR_AN = 252   # jours de bourse par année
RF = 0.03            # taux sans risque annuel (3 %)
np.random.seed(0)    # mêmes résultats à chaque exécution

In [ ]:
actions = ["AAPL", "JPM", "MSFT", "XOM"]
prix = yf.download(actions, period="5y", auto_adjust=True, progress=False)["Close"][actions].dropna()
prix.tail()

## 1. Les prix

In [ ]:
prix.plot(title="Prix des actions", xlabel="Date", ylabel="Prix ($)")

Les prix ne sont pas comparables (une action à 400 $ n'est pas « meilleure » qu'une à 100 $).
On regarde plutôt **ce que devient 1 $ investi**.

In [ ]:
croissance = prix / prix.iloc[0]
croissance.plot(title="Ce que devient 1 $ investi", xlabel="Date", ylabel="Valeur ($)")
plt.axhline(1, color="black", linewidth=1)

## 2. Rendements

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

Exemple : le prix passe de 100 $ à 105 $ → $R = \frac{105 - 100}{100} = +5\,\%$

In [ ]:
rendements = prix.pct_change().dropna()
rendements.head()

In [ ]:
rendements["AAPL"].plot(title="Rendements quotidiens d'AAPL", xlabel="Date", ylabel="Rendement quotidien", linewidth=0.8)
pourcent()

In [ ]:
plt.hist(rendements["AAPL"], bins=100, color=COULEURS[0])
plt.axvline(0, color="black", linewidth=1)
plt.title("Distribution des rendements quotidiens d'AAPL")
plt.xlabel("Rendement quotidien")
plt.ylabel("Nombre de jours")
pourcent("x")

> 💡 La plupart des jours, le prix bouge peu. Quelques rares jours, il bouge beaucoup.

## 3. Log-rendements

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(1 + R_t)$$

**Pourquoi ?** Les rendements simples ne s'additionnent pas dans le temps. Les log-rendements, oui.

Exemple : +10 % puis −10 %.

In [ ]:
print(f"Somme des rendements simples :      {0.10 + (-0.10):+.2%}   ← faux")
print(f"Vrai résultat (100 $ → 110 $ → 99 $) : {1.10 * 0.90 - 1:+.2%}")
print(f"exp(somme des log-rendements) − 1 : {np.exp(np.log(1.10) + np.log(0.90)) - 1:+.2%}   ← exact")

In [ ]:
log_rendements = np.log(prix / prix.shift(1)).dropna()
log_rendements.head()

In [ ]:
R = np.linspace(-0.5, 0.5, 200)
plt.plot(R, R, "--", color="gray", label="Si log = simple")
plt.plot(R, np.log(1 + R), linewidth=3, label="Log-rendement = ln(1 + R)")
plt.scatter(rendements["AAPL"], log_rendements["AAPL"], s=10, color=COULEURS[1], zorder=3, label="Vrais jours d'AAPL")
plt.title("Petits mouvements : simple ≈ log.  Gros mouvements : ils s'écartent.")
plt.xlabel("Rendement simple")
plt.ylabel("Log-rendement")
pourcent("xy")
plt.legend()

**Preuve :** additionner les log-rendements puis appliquer $e^x$ redonne exactement la croissance réelle.

$$\sum_t r_t = \ln\frac{P_T}{P_0} \quad\Rightarrow\quad e^{\sum_t r_t} = \frac{P_T}{P_0}$$

In [ ]:
plt.plot(croissance["AAPL"], linewidth=8, alpha=0.35, label="Avec les prix")
plt.plot(np.exp(log_rendements["AAPL"].cumsum()), "--", color="black", label="exp(somme des log-rendements)")
plt.title("AAPL : les deux courbes sont identiques")
plt.xlabel("Date")
plt.ylabel("Valeur ($)")
plt.legend()

> ✅ **Log-rendements** : s'additionnent **dans le temps** (jour 1 + jour 2 + …)
>
> ✅ **Rendements simples** : s'additionnent **entre actions** (portefeuille = Σ poids × rendement)
>
> Pour la suite (volatilité, corrélation, portefeuilles), on utilise donc les **rendements simples**.

## 4. Volatilité

La volatilité $\sigma$ = l'**écart-type** des rendements. Elle mesure le **risque** : à quel point ça bouge.

$$\sigma_{\text{annuelle}} = \sigma_{\text{quotidienne}} \times \sqrt{252}$$

In [ ]:
vol_quotidienne = rendements.std()
vol_annuelle = vol_quotidienne * np.sqrt(JOURS_PAR_AN)

vol_annuelle.plot(kind="bar", color=COULEURS, rot=0, title="Volatilité annuelle", xlabel="Action", ylabel="Volatilité annuelle")
for i, v in enumerate(vol_annuelle):
    plt.text(i, v, f"{v:.0%}", ha="center", va="bottom", fontsize=15)
pourcent()

In [ ]:
plus, moins = vol_annuelle.idxmax(), vol_annuelle.idxmin()
plt.hist(rendements[plus], bins=100, alpha=0.6, color=COULEURS[1], label=f"{plus} : la plus volatile ({vol_annuelle[plus]:.0%})")
plt.hist(rendements[moins], bins=100, alpha=0.6, color=COULEURS[0], label=f"{moins} : la moins volatile ({vol_annuelle[moins]:.0%})")
plt.title("Plus de volatilité = distribution plus étalée")
plt.xlabel("Rendement quotidien")
plt.ylabel("Nombre de jours")
pourcent("x")
plt.legend()

In [ ]:
r = rendements["AAPL"]
moy, sigma = r.mean(), r.std()
part = ((r > moy - sigma) & (r < moy + sigma)).mean()

plt.hist(r, bins=100, color=COULEURS[0])
plt.axvspan(moy - sigma, moy + sigma, color=COULEURS[3], alpha=0.3, label=f"Entre −1σ et +1σ : {part:.0%} des jours")
plt.title(f"AAPL : σ quotidienne = {sigma:.2%}")
plt.xlabel("Rendement quotidien")
plt.ylabel("Nombre de jours")
pourcent("x")
plt.legend()

In [ ]:
vol_glissante = rendements.rolling(30).std() * np.sqrt(JOURS_PAR_AN)
vol_glissante.plot(title="La volatilité change dans le temps (calculée sur 30 jours glissants)",
                   xlabel="Date", ylabel="Volatilité annuelle")
pourcent()

### Le coût de la volatilité
+10 % puis −10 % = −1 %. Même avec un rendement moyen de 0 %, **la volatilité fait perdre de l'argent**.

$$\text{moyenne des log-rendements} \approx \text{moyenne des rendements simples} - \frac{\sigma^2}{2}$$

In [ ]:
jours = np.arange(250)
for v in [0.01, 0.03, 0.05]:
    hausse_baisse = np.where(jours % 2 == 0, 1 + v, 1 - v)   # +v, −v, +v, −v, ...
    valeur = np.cumprod(hausse_baisse)
    plt.plot(jours[1::2], valeur[1::2], linewidth=3, label=f"+{v:.0%} / −{v:.0%} en alternance")
plt.axhline(1, color="black", linewidth=1)
plt.title("Rendement moyen de 0 %, mais la volatilité coûte cher")
plt.xlabel("Jour")
plt.ylabel("Valeur ($)")
plt.legend()

In [ ]:
moyennes = pd.DataFrame({
    "Moyenne des rendements simples": rendements.mean() * JOURS_PAR_AN,
    "Moyenne des log-rendements": log_rendements.mean() * JOURS_PAR_AN,
})
moyennes.plot(kind="bar", rot=0, title="Rendement annuel moyen : simple vs log", xlabel="Action", ylabel="Rendement annuel")
plt.axhline(0, color="black", linewidth=1)
pourcent()

In [ ]:
ecart = moyennes.iloc[:, 0] - moyennes.iloc[:, 1]
prevu = vol_annuelle ** 2 / 2

plt.scatter(prevu, ecart, s=200, color=COULEURS, zorder=3)
for t in actions:
    plt.annotate(t, (prevu[t], ecart[t]), xytext=(10, -5), textcoords="offset points", fontsize=15)
limite = [0, max(prevu.max(), ecart.max()) * 1.2]
plt.plot(limite, limite, "--", color="gray", label="écart = σ² / 2")
plt.title("Plus une action est volatile, plus l'écart est grand")
plt.xlabel("σ² / 2 (prévu)")
plt.ylabel("Simple − log (observé)")
pourcent("xy")
plt.legend()

## 5. Corrélation

$$\rho_{A,B} = \frac{\text{Cov}(R_A, R_B)}{\sigma_A \, \sigma_B}$$

| ρ | Signification |
|---|---|
| +1 | bougent toujours dans le même sens |
| 0 | aucun lien |
| −1 | bougent toujours en sens opposé |

> 💡 Faible corrélation = quand une action baisse, l'autre ne baisse pas forcément → **diversification**.

In [ ]:
corr = rendements.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(actions)):
    for j in range(len(actions)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=18,
                 color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
plt.xticks(range(len(actions)), actions)
plt.yticks(range(len(actions)), actions)
plt.grid(False)
plt.colorbar(label="Corrélation")
plt.title("Matrice de corrélation")
plt.xlabel("Action")
plt.ylabel("Action")

In [ ]:
# On compare automatiquement la paire la plus corrélée et la moins corrélée
paires = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
haute, basse = paires.idxmax(), paires.idxmin()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (a, b), titre in zip(axes, [haute, basse], ["La plus corrélée", "La moins corrélée"]):
    ax.scatter(rendements[a], rendements[b], s=8, alpha=0.5)
    ax.set_title(f"{titre} : {a} vs {b}  (ρ = {corr.loc[a, b]:.2f})")
    ax.set_xlabel(f"Rendement {a}")
    ax.set_ylabel(f"Rendement {b}")
    pourcent("xy", ax)
plt.tight_layout()

> 💡 Forte corrélation : les points forment une **ligne**. Faible corrélation : un **nuage rond**.

## 6. Markowitz

Un portefeuille = des **poids** $w$ (ex. 60 % AAPL, 40 % XOM), qui totalisent 100 %.

$$\text{Rendement} = w^\top \mu \qquad \text{Risque} = \sqrt{w^\top \Sigma \, w}$$

$\mu$ = rendements moyens annuels, $\Sigma$ = matrice de covariance annuelle.

In [ ]:
mu = rendements.mean() * JOURS_PAR_AN    # rendement annuel moyen
cov = rendements.cov() * JOURS_PAR_AN     # covariance annuelle

### La diversification en une image : on mélange 2 actions

In [ ]:
a, b = basse    # la paire la moins corrélée (section 5)
x = np.linspace(0, 1, 101)    # part de a dans le portefeuille
rend_melange = x * mu[a] + (1 - x) * mu[b]
risque_melange = np.sqrt(x**2 * cov.loc[a, a] + (1 - x)**2 * cov.loc[b, b] + 2 * x * (1 - x) * cov.loc[a, b])
risque_sans_diversification = x * np.sqrt(cov.loc[a, a]) + (1 - x) * np.sqrt(cov.loc[b, b])

plt.plot(risque_sans_diversification, rend_melange, "--", color="gray", label="Si ρ = 1 (aucune diversification)")
plt.plot(risque_melange, rend_melange, linewidth=3, label=f"Vrai mélange (ρ = {corr.loc[a, b]:.2f})")
for p in [0, 0.25, 0.5, 0.75, 1]:
    i = round(p * 100)
    plt.scatter(risque_melange[i], rend_melange[i], s=80, color="black", zorder=3)
    plt.annotate(f"{p:.0%} {a}", (risque_melange[i], rend_melange[i]), xytext=(10, 0), textcoords="offset points")
plt.title(f"Mélanger {a} et {b} réduit le risque")
plt.xlabel("Risque (volatilité annuelle)")
plt.ylabel("Rendement annuel")
pourcent("xy")
plt.legend()

> 💡 L'écart entre la ligne grise et la courbe bleue = **le risque éliminé gratuitement** par la diversification.

### 5 000 portefeuilles aléatoires avec les 4 actions

In [ ]:
poids, rends, risques = [], [], []
for _ in range(5000):
    w = np.random.random(len(actions))
    w = w / w.sum()                      # les poids totalisent 100 %
    poids.append(w)
    rends.append(w @ mu)
    risques.append(np.sqrt(w @ cov @ w))
poids, rends, risques = np.array(poids), np.array(rends), np.array(risques)

sharpes = (rends - RF) / risques         # rendement par unité de risque (section 7)
meilleur = sharpes.argmax()
moins_risque = risques.argmin()

In [ ]:
plt.scatter(risques, rends, c=sharpes, cmap="viridis", s=6)
plt.colorbar(label="Ratio de Sharpe")
plt.scatter(risques[meilleur], rends[meilleur], marker="*", s=600, color="red", edgecolor="black", label="Meilleur Sharpe")
plt.scatter(risques[moins_risque], rends[moins_risque], marker="*", s=600, color="white", edgecolor="black", label="Risque minimal")
for i, t in enumerate(actions):
    plt.scatter(np.sqrt(cov.loc[t, t]), mu[t], marker="X", s=200, color=COULEURS[i], edgecolor="black")
    plt.annotate(t, (np.sqrt(cov.loc[t, t]), mu[t]), xytext=(10, 0), textcoords="offset points", fontsize=15)
plt.title("Chaque point = un portefeuille")
plt.xlabel("Risque (volatilité annuelle)")
plt.ylabel("Rendement annuel")
pourcent("xy")
plt.legend(loc="lower right")

> 💡 Le **bord supérieur gauche** du nuage est la **frontière efficiente** : le meilleur rendement possible pour chaque niveau de risque. Tout point en dessous est inefficace.

In [ ]:
composition = pd.DataFrame({
    f"Meilleur Sharpe (rendement {rends[meilleur]:.1%}, risque {risques[meilleur]:.1%})": poids[meilleur],
    f"Risque minimal (rendement {rends[moins_risque]:.1%}, risque {risques[moins_risque]:.1%})": poids[moins_risque],
}, index=actions)

ax = composition.plot(kind="bar", rot=0, color=["red", "lightgray"], edgecolor="black",
                      title="Composition des 2 portefeuilles spéciaux", xlabel="Action", ylabel="Poids")
for barres in ax.containers:
    ax.bar_label(barres, labels=[f"{v:.0%}" for v in barres.datavalues], fontsize=13)
pourcent()

## 7. Sharpe et Sortino

$$\text{Sharpe} = \frac{R - R_f}{\sigma} \qquad \text{Sortino} = \frac{R - R_f}{\sigma_{\text{baisse}}}$$

- $R_f$ = taux sans risque (ce qu'on gagne sans prendre de risque, ici 3 %)
- **Sharpe** : rendement en plus par unité de risque **total**
- **Sortino** : pareil, mais ne pénalise que les **baisses** (les hausses ne sont pas un « risque »)

> 💡 Plus c'est élevé, mieux c'est. Au-dessus de 1 = bon.

In [ ]:
portefeuilles = {
    "Équipondéré": np.ones(len(actions)) / len(actions),
    "100 % AAPL": np.array([1, 0, 0, 0]),           # ordre : AAPL, JPM, MSFT, XOM
    "Meilleur Sharpe": poids[meilleur],
    "Risque minimal": poids[moins_risque],
}

rend_ptf = pd.DataFrame({nom: rendements @ w for nom, w in portefeuilles.items()})
(1 + rend_ptf).cumprod().plot(title="Ce que devient 1 $ dans chaque portefeuille", xlabel="Date", ylabel="Valeur ($)")
plt.axhline(1, color="black", linewidth=1)

In [ ]:
r = rend_ptf["Équipondéré"]
plt.hist(r[r >= 0], bins=50, color="lightgray", label="Hausses : comptées par Sharpe seulement")
plt.hist(r[r < 0], bins=50, color=COULEURS[1], label="Baisses : comptées par Sharpe ET Sortino")
plt.title("Quels jours comptent comme du risque ?")
plt.xlabel("Rendement quotidien")
plt.ylabel("Nombre de jours")
pourcent("x")
plt.legend()

In [ ]:
rend_annuel = rend_ptf.mean() * JOURS_PAR_AN
vol = rend_ptf.std() * np.sqrt(JOURS_PAR_AN)
vol_baisse = np.sqrt((rend_ptf.clip(upper=0) ** 2).mean()) * np.sqrt(JOURS_PAR_AN)   # seulement les jours négatifs

ratios = pd.DataFrame({
    "Rendement annuel (%)": rend_annuel * 100,
    "Volatilité (%)": vol * 100,
    "Volatilité à la baisse (%)": vol_baisse * 100,
    "Sharpe": (rend_annuel - RF) / vol,
    "Sortino": (rend_annuel - RF) / vol_baisse,
}).round(2)
ratios

In [ ]:
ax = ratios[["Sharpe", "Sortino"]].plot(kind="bar", rot=0, title="Sharpe vs Sortino", xlabel="Portefeuille", ylabel="Ratio")
for barres in ax.containers:
    ax.bar_label(barres, fmt="%.2f", fontsize=13)
plt.axhline(0, color="black", linewidth=1)
plt.axhline(1, color="gray", linestyle="--", label="Seuil « bon » = 1")
plt.ylim(top=max(1.2, ratios[["Sharpe", "Sortino"]].max().max() * 1.15))
plt.legend()

> ⚠️ Le portefeuille « Meilleur Sharpe » a été choisi **avec les mêmes données** que le test. Son score est donc flatteur : rien ne garantit qu'il restera le meilleur dans le futur.

## 📝 Résumé

| Concept | Formule | Code | À retenir |
|---|---|---|---|
| Rendement | $\frac{P_t}{P_{t-1}} - 1$ | `prix.pct_change()` | S'additionne entre actions |
| Log-rendement | $\ln\frac{P_t}{P_{t-1}}$ | `np.log(prix / prix.shift(1))` | S'additionne dans le temps |
| Volatilité | $\sigma \times \sqrt{252}$ | `rendements.std() * np.sqrt(252)` | Mesure le risque, et coûte ≈ σ²/2 par an |
| Corrélation | $\frac{\text{Cov}(A,B)}{\sigma_A \sigma_B}$ | `rendements.corr()` | Faible = meilleure diversification |
| Markowitz | $w^\top\mu$, $\sqrt{w^\top\Sigma w}$ | `w @ mu`, `np.sqrt(w @ cov @ w)` | Combiner réduit le risque |
| Sharpe | $\frac{R - R_f}{\sigma}$ | `(rend - RF) / vol` | Rendement par unité de risque |
| Sortino | $\frac{R - R_f}{\sigma_{\text{baisse}}}$ | `(rend - RF) / vol_baisse` | Ne pénalise que les baisses |